# 09 — Qwen3.5-2B Binary Classification with Unsloth

**Environment:** Google Colab with an NVIDIA CUDA GPU  
**Task:** predict `acute_rejection_within_30_days` from the established synthetic kidney-transplant assessments  
**Scope:** one baseline classifier only; no machine-unlearning method is implemented here

This notebook is intentionally separate from the existing LLM experiments. It establishes whether a Qwen3.5-2B classifier is stable and useful enough to justify a later machine-unlearning experiment.

## Notebook pipeline

1. Clone or update the complete project repository under `/content`.
2. Check the Colab CUDA environment and mount persistent Google Drive storage.
3. Load the frozen dataset, feature contract and split memberships from the GitHub clone.
4. Serialise the same 18 MLP features into deterministic text without the target or identifiers.
5. Load Qwen3.5-2B-Base with Unsloth and add LoRA plus a trainable two-class head.
6. Train on the training split and select the best checkpoint using validation PR-AUC.
7. Freeze a decision threshold using validation data only.
8. Evaluate the test split once and compare with the saved MLP baseline.
9. Save and reload the complete classifier artefacts from Google Drive.

## Load Project from GitHub

**Inputs come from the GitHub clone. Outputs do not.** Edit the two placeholders below once, using the repository that contains the whole `Research Proj` workspace. The normal HTTPS URL is suitable for a public repository.

For a private repository, do not paste a personal access token into this notebook or its URL. Use Colab's Secrets panel or another temporary Git credential method, then keep `GITHUB_REPO_URL` free of credentials.

In [ ]:
from pathlib import Path
import subprocess

# USER CONFIGURATION: replace both <...> placeholders before running this cell.
GITHUB_REPO_URL = "https://github.com/<github-username>/<repo-name>.git"
REPO_ROOT = Path("/content/<repo-name>")
PULL_LATEST = True

if "<" in GITHUB_REPO_URL or "<" in str(REPO_ROOT):
    raise ValueError(
        "Set GITHUB_REPO_URL and REPO_ROOT to your GitHub repository before continuing."
    )

if REPO_ROOT.exists():
    if not (REPO_ROOT / ".git").exists():
        raise RuntimeError(f"{REPO_ROOT} exists but is not a Git clone.")
    if PULL_LATEST:
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
            check=True,
        )
else:
    subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, str(REPO_ROOT)],
        check=True,
    )

SUBMISSION_ROOT = REPO_ROOT / "code/final_submission"
DATA_DIR = SUBMISSION_ROOT / "data/final"
PROCESSED_DIR = SUBMISSION_ROOT / "processed_data"
MLP_RESULTS_DIR = SUBMISSION_ROOT / "results/baseline"

REQUIRED_INPUTS = {
    "Assessment dataset": DATA_DIR / "kidney_transplant_assessments.csv",
    "Feature contract": DATA_DIR / "classifier_feature_list.json",
    "Frozen split assignments": PROCESSED_DIR / "split_assignments.csv",
    "Frozen MLP test metrics": MLP_RESULTS_DIR / "test_metrics.csv",
}
missing_after_clone = [str(path) for path in REQUIRED_INPUTS.values() if not path.is_file()]
if missing_after_clone:
    raise FileNotFoundError(
        "The GitHub clone is missing required project inputs:\n" + "\n".join(missing_after_clone)
    )
print(f"Project repository ready: {REPO_ROOT}")

## 1. Purpose and Research Question

This notebook asks:

> **Can Qwen3.5-2B be fine-tuned efficiently as a binary classifier for acute kidney rejection prediction while maintaining a valid train/validation/test evaluation procedure?**

Class `0` means no confirmed acute rejection within 30 days; class `1` means confirmed acute rejection within 30 days. This is a baseline classification experiment. Full Retraining, Gradient Difference, SISA, retain-set fine-tuning and all other unlearning methods are deferred to a later notebook.

## 2. Relationship to the Existing Project

The main study uses a compact PyTorch MLP on the same synthetic longitudinal dataset. Its established utility measures are PR-AUC, balanced accuracy, binary cross-entropy (BCE), F1 and AUROC. The Qwen experiment is an extension, not a replacement: it reuses the target, approved features and frozen split memberships so that the final comparison is fair.

## 3. Why Qwen3.5-2B?

Qwen3.5-2B follows the supervisor's requested model family and size. Two billion parameters provide a meaningful LLM experiment while remaining feasible on a Colab NVIDIA GPU with parameter-efficient training. The Base variant is used because this experiment learns a new supervised classification head; instruction-following or chatbot behaviour is neither needed nor evaluated.

## 4. Why Google Colab and Unsloth?

Local Apple MPS training is not the intended environment. Colab supplies CUDA-capable NVIDIA hardware, while Unsloth reduces the memory and compute cost of LoRA fine-tuning. [Current Qwen3.5 guidance](https://unsloth.ai/docs/models/qwen3.5/fine-tune) recommends 16-bit LoRA rather than 4-bit QLoRA because Qwen3.5 can show larger quantisation differences. The notebook therefore fails if CUDA is absent and never falls back to CPU or MPS.

## 5. Classification Architecture

The design adapts the final-token classification principle in [gpjt/qwen-classifier](https://github.com/gpjt/qwen-classifier), not its spam dataset:

```text
serialised assessment
        ↓
     tokenizer
        ↓
 Qwen3.5-2B-Base + LoRA
        ↓
two logits at the final non-padding token
        ↓
 softmax probability → class 0 or class 1
```

The language-model output projection is replaced by a trainable linear layer with exactly two outputs. Cross-entropy is calculated directly from those two logits. Qwen is therefore a classifier, not a chatbot prompted to generate the text `0` or `1`.

## 6. Colab Environment Setup

The installation below follows the current official Qwen3.5-2B Unsloth Colab recipe. The Transformer version is pinned because Qwen3.5 requires the Transformers 5 architecture; the pin is a compatibility requirement rather than an arbitrary historical choice. Package installation and initial Qwen3.5 kernel compilation can take several minutes on a T4.

In [ ]:
# Run this cell once in a fresh Colab GPU runtime.
%pip install -q --upgrade uv
!uv pip install -q "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes "xformers==0.0.32.post2" pandas numpy scikit-learn matplotlib seaborn datasets peft safetensors
!uv pip install -q "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"
!uv pip install -q "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install -q --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" unsloth unsloth_zoo
!uv pip install -q "transformers==5.2.0"
!uv pip install -q --no-build-isolation "flash-linear-attention" "causal_conv1d==1.6.0"

# The official recipe disables TileLang kernels on pre-Ampere GPUs such as a T4.
import os as _os
import torch as _torch
if _torch.cuda.is_available() and _torch.cuda.get_device_capability()[0] < 8:
    _os.environ["FLA_TILELANG"] = "0"

In [ ]:
import importlib.metadata
import platform
import sys

import torch
import transformers


def show_gpu_information() -> dict:
    "Return and print the runtime versions and CUDA hardware."
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. In Colab choose Runtime > Change runtime type > NVIDIA GPU, "
            "then restart and run from the top. CPU and MPS fallback are not allowed."
        )
    properties = torch.cuda.get_device_properties(0)
    details = {
        "Python": sys.version.split()[0],
        "PyTorch": torch.__version__,
        "Transformers": transformers.__version__,
        "Unsloth": importlib.metadata.version("unsloth"),
        "Platform": platform.platform(),
        "CUDA available": True,
        "GPU": properties.name,
        "GPU memory (GiB)": round(properties.total_memory / 1024**3, 2),
    }
    return details


environment_info = show_gpu_information()
display(pd.Series(environment_info, name="Value").to_frame()) if "pd" in globals() else print(environment_info)
DEVICE = torch.device("cuda")

## 7. Mount Google Drive

Colab runtime storage is temporary. Project inputs remain in the GitHub clone under `REPO_ROOT`; Google Drive is mounted only to persist Qwen checkpoints, adapters and run results.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")  # User interaction: approve access to your own Drive.

In [ ]:
from datetime import datetime, timezone

# USER CONFIGURATION: this Drive folder stores Qwen outputs, never project inputs.
QWEN_OUTPUT_ROOT = Path("/content/drive/MyDrive/Qwen Experiment")
MODEL_ROOT = QWEN_OUTPUT_ROOT / "models"
RESULT_ROOT = QWEN_OUTPUT_ROOT / "results"
CHECKPOINT_ROOT = QWEN_OUTPUT_ROOT / "checkpoints"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RUN_MODEL_DIR = MODEL_ROOT / RUN_ID
RUN_RESULT_DIR = RESULT_ROOT / RUN_ID
BEST_ADAPTER_DIR = CHECKPOINT_ROOT / RUN_ID / "best_adapter"
for path in [RUN_MODEL_DIR, RUN_RESULT_DIR, BEST_ADAPTER_DIR.parent]:
    path.mkdir(parents=True, exist_ok=True)

paths = {
    "GitHub inputs": SUBMISSION_ROOT,
    "Drive models": RUN_MODEL_DIR,
    "Drive results": RUN_RESULT_DIR,
    "Drive checkpoints": BEST_ADAPTER_DIR.parent,
}
display(pd.Series({k: str(v) for k, v in paths.items()}, name="Path").to_frame()) if "pd" in globals() else print(paths)

## 8. Reproducibility

Random sources are seeded before data loading and training. CUDA operations can still vary across drivers, kernels and GPU models, so this improves repeatability without claiming perfect bit-for-bit determinism.

In [ ]:
import json
import os
import random
import time

import numpy as np
import pandas as pd

SEED = 42


def set_reproducible_seed(seed: int) -> None:
    "Seed Python, NumPy and PyTorch without promising perfect GPU determinism."
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_reproducible_seed(SEED)
print(f"Random seed: {SEED}")

## 9. Load the Established Dataset

No records are generated here and no remote inference API is called. The input is the immutable synthetic assessment table in the GitHub clone. Identifiers are used only to verify joins and split membership; they are not displayed as model inputs.

In [ ]:
ASSESSMENT_PATH = REQUIRED_INPUTS["Assessment dataset"]
FEATURE_CONTRACT_PATH = REQUIRED_INPUTS["Feature contract"]
SPLIT_PATH = REQUIRED_INPUTS["Frozen split assignments"]
MLP_METRICS_PATH = REQUIRED_INPUTS["Frozen MLP test metrics"]

input_verification = pd.DataFrame([
    {
        "Artefact": artefact,
        "Repository Path": path.relative_to(REPO_ROOT).as_posix(),
        "Exists?": path.is_file(),
    }
    for artefact, path in REQUIRED_INPUTS.items()
])
display(input_verification)

missing_inputs = [str(path) for path in REQUIRED_INPUTS.values() if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "Required project files are missing from the GitHub clone:\n" + "\n".join(missing_inputs)
    )

assessments = pd.read_csv(ASSESSMENT_PATH)
feature_contract = json.loads(FEATURE_CONTRACT_PATH.read_text(encoding="utf-8"))
TARGET = feature_contract["target"]

dataset_summary = {
    "assessment rows": len(assessments),
    "columns": assessments.shape[1],
    "recipients": assessments["recipient_id"].nunique(),
    "class 0": int((assessments[TARGET] == 0).sum()),
    "class 1": int((assessments[TARGET] == 1).sum()),
    "positive prevalence": float(assessments[TARGET].mean()),
}
display(pd.Series(dataset_summary, name="Value").to_frame())

## 10. Reuse the Frozen Train / Validation / Test Split

The saved membership file is joined by `recipient_id`; no random split function is called. Identical memberships are necessary because changing recipients or donors between partitions would confound comparison with the MLP.

In [ ]:
split_assignments = pd.read_csv(SPLIT_PATH)
assert split_assignments["recipient_id"].is_unique
assert set(split_assignments["split"]) == {"train", "validation", "test"}

data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)
assert data["split"].notna().all()
assert len(data) == len(assessments) == 60_000

split_frames = {
    name: data.loc[data["split"].eq(name)].reset_index(drop=True)
    for name in ["train", "validation", "test"]
}

expected_rows = {"train": 42_024, "validation": 8_988, "test": 8_988}
assert {name: len(frame) for name, frame in split_frames.items()} == expected_rows
assert sum(expected_rows.values()) == len(data)

for key in ["assessment_id", "recipient_id", "donor_id"]:
    memberships = {name: set(frame[key]) for name, frame in split_frames.items()}
    assert memberships["train"].isdisjoint(memberships["validation"])
    assert memberships["train"].isdisjoint(memberships["test"])
    assert memberships["validation"].isdisjoint(memberships["test"])

assert data.groupby("recipient_id")["split"].nunique().max() == 1
assert data.groupby("donor_id")["split"].nunique().max() == 1
ORIGINAL_SPLITS_REUSED = True

split_summary = pd.DataFrame([
    {
        "split": name,
        "assessments": len(frame),
        "recipients": frame["recipient_id"].nunique(),
        "donors": frame["donor_id"].nunique(),
        "positives": int(frame[TARGET].sum()),
        "prevalence": frame[TARGET].mean(),
    }
    for name, frame in split_frames.items()
])
display(split_summary)

## 11. Define the Approved Input Features

The JSON feature contract is authoritative. Qwen receives exactly the same 18 predictive fields as the MLP, only encoded as text. Identifiers, consent/retention fields, dates, hospital information and the target are blocked explicitly.

In [ ]:
FEATURES = feature_contract["classifier_features"]
EXPECTED_FEATURES = [
    "recipient_age", "donor_age", "donor_type", "kidney_failure_cause",
    "previous_transplant", "dialysis_months", "abo_compatibility_category",
    "hla_mismatch_count", "antibody_risk_score", "cold_ischaemia_hours",
    "days_since_transplant", "creatinine_mg_dl", "creatinine_change_pct",
    "urine_output_ml_24h", "tacrolimus_level_ng_ml",
    "medication_adherence_pct", "infection_indicator", "previous_rejection",
]
BLOCKED_COLUMNS = {
    "assessment_id", "recipient_id", "donor_id", "hospital_id", "assessment_date",
    "training_consent_status", "training_consent_version", "retention_expiry_date", TARGET,
}

assert FEATURES == EXPECTED_FEATURES
assert len(FEATURES) == 18
assert set(FEATURES).isdisjoint(BLOCKED_COLUMNS)
assert set(FEATURES).issubset(assessments.columns)
display(pd.DataFrame({"position": range(1, len(FEATURES) + 1), "approved feature": FEATURES}))

## 12. Deterministic Tabular-to-Text Serialisation

A language model needs a token sequence, so each row is converted to short field-value statements in one fixed order. The serializer adds no clinical interpretation. The target remains separate, and identifiers never enter the text.

In [ ]:
FEATURE_LABELS = {
    "recipient_age": "Recipient age (years)",
    "donor_age": "Donor age (years)",
    "donor_type": "Donor type",
    "kidney_failure_cause": "Kidney failure cause",
    "previous_transplant": "Previous transplant",
    "dialysis_months": "Dialysis duration (months)",
    "abo_compatibility_category": "ABO compatibility",
    "hla_mismatch_count": "HLA mismatch count",
    "antibody_risk_score": "Antibody risk score",
    "cold_ischaemia_hours": "Cold ischaemia time (hours)",
    "days_since_transplant": "Days since transplant",
    "creatinine_mg_dl": "Creatinine (mg/dL)",
    "creatinine_change_pct": "Creatinine change (percent)",
    "urine_output_ml_24h": "Urine output (mL/24h)",
    "tacrolimus_level_ng_ml": "Tacrolimus level (ng/mL)",
    "medication_adherence_pct": "Medication adherence (percent)",
    "infection_indicator": "Infection indicator",
    "previous_rejection": "Previous rejection",
}
BINARY_FEATURES = {"previous_transplant", "infection_indicator", "previous_rejection"}
SERIALISATION_VERSION = "kidney-qwen35-v1"


def format_feature_value(feature: str, value) -> str:
    "Format one stored feature deterministically without adding information."
    if pd.isna(value):
        return "missing"
    if feature in BINARY_FEATURES:
        return "yes" if int(value) == 1 else "no"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}".rstrip("0").rstrip(".")
    return str(value).strip()


def serialize_assessment(row: pd.Series) -> str:
    "Serialise exactly the approved features in their frozen order."
    return "\n".join(
        f"{FEATURE_LABELS[feature]}: {format_feature_value(feature, row[feature])}."
        for feature in FEATURES
    )


for name, frame in split_frames.items():
    frame["text"] = frame.apply(serialize_assessment, axis=1)

NO_TARGET_LEAKAGE = all(
    TARGET not in text
    and all(blocked not in text for blocked in BLOCKED_COLUMNS)
    for frame in split_frames.values()
    for text in frame["text"]
)
assert NO_TARGET_LEAKAGE

example_zero = split_frames["train"].loc[split_frames["train"][TARGET].eq(0), "text"].iloc[0]
example_one = split_frames["train"].loc[split_frames["train"][TARGET].eq(1), "text"].iloc[0]
print("CLASS 0 EXAMPLE\n", example_zero, "\n\nCLASS 1 EXAMPLE\n", example_one)

## 13. Prepare Classification Labels

Labels are integer tensors stored separately from text. The model is trained to minimise two-class cross-entropy, not to generate a label string.

In [ ]:
for name, frame in split_frames.items():
    assert set(frame[TARGET].unique()).issubset({0, 1})
    frame["label"] = frame[TARGET].astype("int64")

assert TARGET not in FEATURES
print("Labels:", {0: "no acute rejection within 30 days", 1: "acute rejection within 30 days"})

## 14. Tokenisation

Sequence length is selected from training text only. The maximum observed training length is rounded up to a multiple of eight, with a 512-token safety ceiling. Validation and test text can be measured for transparent truncation reporting, but their labels do not affect this choice.

In [ ]:
from transformers import AutoProcessor

MODEL_ID = "unsloth/Qwen3.5-2B-Base"

# Load only the processor for length inspection; model weights are not needed yet.
processor = AutoProcessor.from_pretrained(MODEL_ID)
text_tokenizer = getattr(processor, "tokenizer", processor)
text_tokenizer.padding_side = "right"
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token


def token_lengths(texts: list[str], batch_size: int = 512) -> np.ndarray:
    "Measure untruncated token counts in bounded batches."
    lengths = []
    for start in range(0, len(texts), batch_size):
        encoded = text_tokenizer(
            texts[start:start + batch_size], add_special_tokens=True,
            truncation=False, padding=False,
        )["input_ids"]
        lengths.extend(map(len, encoded))
    return np.asarray(lengths)


train_lengths = token_lengths(split_frames["train"]["text"].tolist())
MAX_SEQ_LENGTH = min(512, int(np.ceil(train_lengths.max() / 8) * 8))

length_rows = []
for name, frame in split_frames.items():
    lengths = train_lengths if name == "train" else token_lengths(frame["text"].tolist())
    frame["token_length"] = lengths
    length_rows.append({
        "split": name,
        "minimum": int(lengths.min()),
        "median": float(np.median(lengths)),
        "95th percentile": float(np.percentile(lengths, 95)),
        "maximum": int(lengths.max()),
        "truncated": int((lengths > MAX_SEQ_LENGTH).sum()),
    })

display(pd.DataFrame(length_rows))
print(f"Chosen maximum sequence length: {MAX_SEQ_LENGTH} tokens (derived from training text only)")

## 15. Load Qwen3.5-2B with Unsloth

The exact Base checkpoint is loaded into the Colab runtime in 16-bit mode. No Qwen API, alternate model or MPS device is used. The visual encoder remains frozen because the input is text only.

In [ ]:
# Release the temporary processor reference before the full model is created.
del processor
torch.cuda.empty_cache()

from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=False,
    load_in_16bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
    use_gradient_checkpointing="unsloth",
)
text_tokenizer = getattr(processor, "tokenizer", processor)
text_tokenizer.padding_side = "right"
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

assert "Qwen3.5-2B" in MODEL_ID
assert next(model.parameters()).is_cuda
print("Loaded model:", MODEL_ID)

## 16. Add Parameter-Efficient Fine-Tuning

Qwen3.5 combines full-attention and gated linear-attention layers. Restricting LoRA to only `q_proj` and `v_proj` would miss much of this hybrid backbone. Current Unsloth guidance therefore supports `all-linear`; the classification head is excluded from LoRA and stored as a fully trainable module. Rank 16, alpha 16, zero LoRA dropout and Unsloth gradient checkpointing form a conservative first baseline.

## 17. Create the Binary Classification Head

The original vocabulary projection is replaced before LoRA is attached. `modules_to_save=["lm_head"]` instructs PEFT to train and persist this non-LoRA module with the adapter. A real forward pass below verifies the required `[batch_size, 2]` output.

In [ ]:
from torch import nn

original_output_head = model.get_output_embeddings()
hidden_size = original_output_head.in_features
head_device = original_output_head.weight.device
head_dtype = original_output_head.weight.dtype

binary_head = nn.Linear(hidden_size, 2, bias=False, device=head_device, dtype=head_dtype)
nn.init.normal_(binary_head.weight, mean=0.0, std=0.02)
model.set_output_embeddings(binary_head)
model.config.num_labels = 2
model.config.pad_token_id = text_tokenizer.pad_token_id
del original_output_head
torch.cuda.empty_cache()

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
    modules_to_save=["lm_head"],
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.config.use_cache = False

In [ ]:
smoke_text = split_frames["train"]["text"].iloc[:2].tolist()
smoke_inputs = text_tokenizer(
    smoke_text, padding=True, truncation=True,
    max_length=MAX_SEQ_LENGTH, return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    smoke_sequence_logits = model(**smoke_inputs).logits
smoke_last_indices = smoke_inputs["attention_mask"].sum(dim=1) - 1
smoke_logits = smoke_sequence_logits[
    torch.arange(len(smoke_text), device=DEVICE), smoke_last_indices
]

assert smoke_logits.shape == (2, 2)
assert torch.isfinite(smoke_logits).all()
LOGITS_SHAPE_VERIFIED = True
print("Final-token logits shape:", tuple(smoke_logits.shape))
del smoke_inputs, smoke_sequence_logits, smoke_logits

## 18. Inspect Trainable Parameters

Only LoRA adapter weights and the two-class head should update. The visual encoder and original Qwen weights remain frozen.

In [ ]:
parameter_rows = []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        family = "classification head" if "lm_head" in name else "LoRA" if "lora_" in name else "other"
        parameter_rows.append({"name": name, "parameters": parameter.numel(), "family": family})

trainable_parameters = sum(row["parameters"] for row in parameter_rows)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
frozen_parameters = total_parameters - trainable_parameters
head_was_trainable = any(row["family"] == "classification head" for row in parameter_rows)
assert head_was_trainable
assert all(row["family"] in {"classification head", "LoRA"} for row in parameter_rows)

parameter_summary = pd.DataFrame([
    {"category": "total", "parameters": total_parameters},
    {"category": "trainable", "parameters": trainable_parameters},
    {"category": "frozen", "parameters": frozen_parameters},
    {"category": "LoRA trainable", "parameters": sum(r["parameters"] for r in parameter_rows if r["family"] == "LoRA")},
    {"category": "head trainable", "parameters": sum(r["parameters"] for r in parameter_rows if r["family"] == "classification head")},
])
parameter_summary["percent of total"] = 100 * parameter_summary["parameters"] / total_parameters
display(parameter_summary)

## 19. Class Imbalance Handling

Acute rejection is uncommon. Class weights are calculated from training labels only and used only in the training loss. Validation and test BCE remain unweighted so they describe probability quality on the observed distributions.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

y_train = split_frames["train"]["label"].to_numpy()
class_weight_values = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_train)
CLASS_WEIGHTS = torch.tensor(class_weight_values, dtype=torch.float32, device=DEVICE)
display(pd.DataFrame({"class": [0, 1], "training weight": class_weight_values}))

## 20. Training Configuration

This is one conservative baseline, not a hyperparameter search. A physical batch of four and eight accumulation steps give an effective batch of 32. Validation PR-AUC selects the checkpoint because it is informative under class imbalance.

In [ ]:
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 2e-5
MAX_EPOCHS = 3
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2
GRADIENT_CLIP_NORM = 1.0

training_configuration = {
    "model": MODEL_ID,
    "batch size": BATCH_SIZE,
    "effective batch size": EFFECTIVE_BATCH_SIZE,
    "learning rate": LEARNING_RATE,
    "maximum epochs": MAX_EPOCHS,
    "LoRA rank": 16,
    "LoRA alpha": 16,
    "LoRA dropout": 0,
    "LoRA targets": "all linear language layers (hybrid Qwen3.5 coverage)",
    "maximum sequence length": MAX_SEQ_LENGTH,
    "optimiser": "AdamW",
    "weight decay": WEIGHT_DECAY,
    "early stopping patience": EARLY_STOPPING_PATIENCE,
    "selection metric": "validation PR-AUC (higher is better)",
    "random seed": SEED,
}
display(pd.Series(training_configuration, name="Value").to_frame())

## 21. Baseline Training

The data loader tokenises with dynamic right padding to avoid computing over unnecessary padding tokens. The training loop never receives the test frame or test loader. At each epoch it records weighted training loss, unweighted validation loss and validation PR-AUC, then saves a new best adapter and head to Drive.

In [ ]:
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset


class AssessmentTextDataset(Dataset):
    "Minimal text/label dataset; identifiers are returned only for saved predictions."

    def __init__(self, frame: pd.DataFrame):
        self.texts = frame["text"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.assessment_ids = frame["assessment_id"].astype(str).tolist()

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict:
        return {
            "text": self.texts[index],
            "label": self.labels[index],
            "assessment_id": self.assessment_ids[index],
        }


def collate_assessments(rows: list[dict]) -> dict:
    "Tokenise one batch without putting labels into its input text."
    encoded = text_tokenizer(
        [row["text"] for row in rows],
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": torch.tensor([row["label"] for row in rows], dtype=torch.long),
        "assessment_id": [row["assessment_id"] for row in rows],
    }


generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    AssessmentTextDataset(split_frames["train"]), batch_size=BATCH_SIZE,
    shuffle=True, generator=generator, collate_fn=collate_assessments, num_workers=0,
)
validation_loader = DataLoader(
    AssessmentTextDataset(split_frames["validation"]), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_assessments, num_workers=0,
)

# The test loader is deliberately not created until Section 24.
TEST_EVALUATION_STARTED = False
TEST_ROWS_USED_FOR_TRAINING = 0

In [ ]:
from sklearn.metrics import average_precision_score


def final_token_logits(model_object, batch: dict) -> torch.Tensor:
    "Return two logits at each sequence's final non-padding token."
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    sequence_logits = model_object(input_ids=input_ids, attention_mask=attention_mask).logits
    final_indices = attention_mask.sum(dim=1) - 1
    logits = sequence_logits[torch.arange(input_ids.shape[0], device=DEVICE), final_indices]
    if logits.shape != (input_ids.shape[0], 2):
        raise RuntimeError(f"Expected [batch, 2] logits, received {tuple(logits.shape)}")
    return logits


@torch.no_grad()
def evaluate_classifier(model_object, loader: DataLoader) -> dict:
    "Calculate unweighted loss and probabilities without changing model state."
    model_object.eval()
    losses, labels, probabilities, identifiers, logits_rows = [], [], [], [], []
    for batch in loader:
        batch_labels = batch["labels"].to(DEVICE)
        logits = final_token_logits(model_object, batch)
        losses.append(F.cross_entropy(logits.float(), batch_labels, reduction="sum").item())
        labels.extend(batch_labels.cpu().numpy())
        probabilities.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        logits_rows.extend(logits.float().cpu().numpy())
        identifiers.extend(batch["assessment_id"])
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    return {
        "loss": float(sum(losses) / len(labels)),
        "pr_auc": float(average_precision_score(labels, probabilities)),
        "labels": labels,
        "probabilities": probabilities,
        "logits": np.asarray(logits_rows),
        "assessment_ids": identifiers,
    }

In [ ]:
from math import ceil
from transformers import get_linear_schedule_with_warmup

trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimiser = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
updates_per_epoch = ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_updates = updates_per_epoch * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimiser, num_warmup_steps=max(1, int(0.05 * total_updates)), num_training_steps=total_updates,
)

USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=not USE_BF16)

fixed_probe = next(iter(validation_loader))
history_rows = []
best_validation_pr_auc = -np.inf
best_validation_loss = np.inf
best_epoch = None
best_probe_probabilities = None
epochs_without_improvement = 0

torch.cuda.reset_peak_memory_stats()
training_started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimiser.zero_grad(set_to_none=True)
    running_loss = 0.0
    examples_seen = 0
    epoch_started = time.perf_counter()

    for step, batch in enumerate(train_loader):
        labels = batch["labels"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE):
            logits = final_token_logits(model, batch)
            loss = F.cross_entropy(logits.float(), labels, weight=CLASS_WEIGHTS)
            scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

        if not torch.isfinite(loss):
            raise FloatingPointError("Training loss became non-finite.")
        scaler.scale(scaled_loss).backward()
        running_loss += loss.item() * len(labels)
        examples_seen += len(labels)

        update_now = (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader)
        if update_now:
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(trainable, GRADIENT_CLIP_NORM)
            scaler.step(optimiser)
            scaler.update()
            scheduler.step()
            optimiser.zero_grad(set_to_none=True)

    validation_result = evaluate_classifier(model, validation_loader)
    epoch_seconds = time.perf_counter() - epoch_started
    history_rows.append({
        "epoch": epoch,
        "training_loss": running_loss / examples_seen,
        "validation_loss": validation_result["loss"],
        "validation_pr_auc": validation_result["pr_auc"],
        "epoch_seconds": epoch_seconds,
    })

    improved = (
        validation_result["pr_auc"] > best_validation_pr_auc + 1e-12
        or (
            np.isclose(validation_result["pr_auc"], best_validation_pr_auc)
            and validation_result["loss"] < best_validation_loss
        )
    )
    if improved:
        best_validation_pr_auc = validation_result["pr_auc"]
        best_validation_loss = validation_result["loss"]
        best_epoch = epoch
        epochs_without_improvement = 0
        model.save_pretrained(BEST_ADAPTER_DIR, safe_serialization=True)
        processor.save_pretrained(BEST_ADAPTER_DIR)

        # Keep a small prediction reference from the exact state just saved.
        model.eval()
        with torch.no_grad():
            best_probe_probabilities = torch.softmax(
                final_token_logits(model, fixed_probe).float(), dim=1
            )[:, 1].cpu().numpy()

        head_state = {
            name: tensor.detach().cpu()
            for name, tensor in model.state_dict().items()
            if "lm_head" in name
        }
        assert head_state, "The classification head was not found in the saved model state."
        torch.save(head_state, RUN_MODEL_DIR / "binary_classification_head.pt")
    else:
        epochs_without_improvement += 1

    print(history_rows[-1])
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping after epoch {epoch}; best epoch was {best_epoch}.")
        break

training_seconds = time.perf_counter() - training_started
peak_gpu_memory_gib = torch.cuda.max_memory_allocated() / 1024**3
training_history = pd.DataFrame(history_rows)
training_history.to_csv(RUN_RESULT_DIR / "training_history.csv", index=False)

assert best_epoch is not None and BEST_ADAPTER_DIR.exists()
assert TEST_ROWS_USED_FOR_TRAINING == 0 and not TEST_EVALUATION_STARTED

## 22. Training History

The selected epoch is the one with the highest validation PR-AUC; validation loss breaks an exact tie. The following output should be interpreted cautiously: decreasing training loss with worsening validation results would suggest overfitting.

In [ ]:
import matplotlib.pyplot as plt

display(training_history)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, column, title in zip(
    axes,
    ["training_loss", "validation_loss", "validation_pr_auc"],
    ["Training loss", "Validation loss", "Validation PR-AUC"],
):
    axis.plot(training_history["epoch"], training_history[column], marker="o")
    axis.axvline(best_epoch, color="tab:green", linestyle="--", alpha=0.7, label="selected epoch")
    axis.set(xlabel="Epoch", title=title)
    axis.legend()
fig.tight_layout()
plt.show()

last_epoch = int(training_history["epoch"].iloc[-1])
overfit_signal = (
    last_epoch > best_epoch
    and training_history.loc[training_history["epoch"].eq(last_epoch), "validation_loss"].iloc[0]
    > best_validation_loss
)
display(Markdown(
    f"The selected checkpoint is epoch **{best_epoch}**, with validation PR-AUC "
    f"**{best_validation_pr_auc:.4f}** and validation loss **{best_validation_loss:.4f}**. "
    f"A simple late-epoch overfitting signal was {'present' if overfit_signal else 'not present'}; "
    "the short run does not support stronger claims."
))

### Reload the selected checkpoint before threshold selection

The last training epoch need not be the best epoch. The in-memory training model is released, then the saved Base + adapter + head is reconstructed. Predictions on a fixed validation probe must match those recorded when the checkpoint was written.

In [ ]:
from peft import PeftModel


def load_saved_classifier(adapter_path: Path):
    "Rebuild Qwen3.5 with a two-logit head, then load the saved PEFT state."
    base_model, saved_processor = FastVisionModel.from_pretrained(
        MODEL_ID,
        load_in_4bit=False,
        load_in_16bit=True,
        max_seq_length=MAX_SEQ_LENGTH,
        use_gradient_checkpointing="unsloth",
    )
    old_head = base_model.get_output_embeddings()
    reloaded_head = nn.Linear(
        old_head.in_features, 2, bias=False,
        device=old_head.weight.device, dtype=old_head.weight.dtype,
    )
    base_model.set_output_embeddings(reloaded_head)
    base_model.config.num_labels = 2
    base_model.config.pad_token_id = getattr(saved_processor, "tokenizer", saved_processor).pad_token_id
    classifier = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=False)
    classifier.config.use_cache = False
    return classifier, saved_processor


del model, optimiser, scheduler, scaler, trainable
torch.cuda.empty_cache()

model, processor = load_saved_classifier(BEST_ADAPTER_DIR)
text_tokenizer = getattr(processor, "tokenizer", processor)
text_tokenizer.padding_side = "right"
model.eval()

with torch.no_grad():
    reloaded_probe_probabilities = torch.softmax(
        final_token_logits(model, fixed_probe).float(), dim=1
    )[:, 1].cpu().numpy()

reload_max_absolute_difference = float(
    np.max(np.abs(reloaded_probe_probabilities - best_probe_probabilities))
)
RELOAD_VERIFIED = np.allclose(
    reloaded_probe_probabilities, best_probe_probabilities, rtol=1e-4, atol=1e-5
)
assert RELOAD_VERIFIED
print(f"Reload verification maximum probability difference: {reload_max_absolute_difference:.8f}")

## 23. Validation Threshold Selection

The best checkpoint is now fixed. The MLP's established threshold rule is reused: maximise validation F1, then balanced accuracy, then proximity to 0.5, then the lower threshold. No test prediction is available at this stage.

In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
)

validation_result = evaluate_classifier(model, validation_loader)
validation_threshold_rows = []
for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2):
    predictions = (validation_result["probabilities"] >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(validation_result["labels"], predictions, labels=[0, 1]).ravel()
    validation_threshold_rows.append({
        "threshold": float(threshold),
        "precision": precision_score(validation_result["labels"], predictions, zero_division=0),
        "recall": recall_score(validation_result["labels"], predictions, zero_division=0),
        "f1": f1_score(validation_result["labels"], predictions, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(validation_result["labels"], predictions),
        "specificity": tn / (tn + fp),
        "distance_from_0_5": abs(float(threshold) - 0.5),
    })

validation_thresholds = pd.DataFrame(validation_threshold_rows)
selection_order = validation_thresholds.sort_values(
    ["f1", "balanced_accuracy", "distance_from_0_5", "threshold"],
    ascending=[False, False, True, True],
    kind="mergesort",
)
SELECTED_THRESHOLD = float(selection_order.iloc[0]["threshold"])
validation_thresholds["selected"] = validation_thresholds["threshold"].eq(SELECTED_THRESHOLD)
validation_thresholds.to_csv(RUN_RESULT_DIR / "validation_thresholds.csv", index=False)
(RUN_RESULT_DIR / "selected_threshold.json").write_text(
    json.dumps({"threshold": SELECTED_THRESHOLD, "source": "validation only"}, indent=2),
    encoding="utf-8",
)
display(selection_order.head(10))
print(f"Frozen validation-selected threshold: {SELECTED_THRESHOLD:.2f}")
assert not TEST_EVALUATION_STARTED

## 24. Final Test Evaluation

Only now, after the checkpoint, hyperparameters and threshold are fixed, is the test loader created. The test split is evaluated once. No later cell changes the trained model or threshold.

In [ ]:
def compute_metrics(labels: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    "Return the project's binary utility metrics from fixed probabilities."
    predictions = (probabilities >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "n": int(len(labels)),
        "positive_count": int(labels.sum()),
        "prevalence": float(labels.mean()),
        "threshold": float(threshold),
        "pr_auc": float(average_precision_score(labels, probabilities)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "binary_cross_entropy": float(log_loss(labels, probabilities, labels=[0, 1])),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "auroc": float(roc_auc_score(labels, probabilities)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "specificity": float(tn / (tn + fp)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


TEST_EVALUATION_STARTED = True
test_loader = DataLoader(
    AssessmentTextDataset(split_frames["test"]), batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_assessments, num_workers=0,
)
test_result = evaluate_classifier(model, test_loader)
test_metrics = compute_metrics(
    test_result["labels"], test_result["probabilities"], SELECTED_THRESHOLD
)
display(pd.Series(test_metrics, name="Qwen3.5-2B test value").to_frame())

test_predictions = pd.DataFrame({
    "assessment_id": test_result["assessment_ids"],
    "label": test_result["labels"],
    "logit_0": test_result["logits"][:, 0],
    "logit_1": test_result["logits"][:, 1],
    "probability_class_1": test_result["probabilities"],
    "prediction": (test_result["probabilities"] >= SELECTED_THRESHOLD).astype(int),
})
test_predictions.to_csv(RUN_RESULT_DIR / "test_predictions.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(RUN_RESULT_DIR / "test_metrics.csv", index=False)

In [ ]:
confusion = np.array([
    [test_metrics["true_negative"], test_metrics["false_positive"]],
    [test_metrics["false_negative"], test_metrics["true_positive"]],
])
fig, ax = plt.subplots(figsize=(4.5, 4))
image = ax.imshow(confusion, cmap="Blues")
for (row, column), value in np.ndenumerate(confusion):
    ax.text(column, row, str(value), ha="center", va="center")
ax.set(
    xticks=[0, 1], yticks=[0, 1],
    xlabel="Predicted class", ylabel="Observed class",
    title=f"Qwen3.5-2B test confusion matrix (threshold={SELECTED_THRESHOLD:.2f})",
)
fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()

## 25. Comparison with the Existing MLP

The MLP is not retrained. Its frozen test metrics are loaded from the existing baseline artefact and aligned with the Qwen outputs. Differences are reported neutrally; a larger language model is not assumed to be better for tabular data.

In [ ]:
mlp_metrics = pd.read_csv(MLP_METRICS_PATH).iloc[0]
metric_labels = {
    "pr_auc": "PR-AUC",
    "balanced_accuracy": "Balanced Accuracy",
    "binary_cross_entropy": "BCE",
    "f1": "F1",
    "auroc": "AUROC",
}
comparison = pd.DataFrame([
    {
        "Metric": label,
        "MLP": float(mlp_metrics[key]),
        "Qwen3.5-2B": float(test_metrics[key]),
    }
    for key, label in metric_labels.items()
])
comparison.to_csv(RUN_RESULT_DIR / "mlp_qwen_test_comparison.csv", index=False)
display(comparison)

qwen_wins = int(sum(
    (row["Qwen3.5-2B"] > row["MLP"]) if row["Metric"] != "BCE"
    else (row["Qwen3.5-2B"] < row["MLP"])
    for _, row in comparison.iterrows()
))
display(Markdown(
    f"Qwen has the favourable value on **{qwen_wins} of 5** reported utility metrics. "
    "This comparison concerns predictive utility only; the Qwen model is substantially larger and more expensive than the MLP."
))

## 26. Runtime and Resource Use

Runtime and peak allocated CUDA memory describe this Colab run, not a universal hardware benchmark. They support the later decision about whether the 2B classifier's cost is justified.

In [ ]:
resource_summary = {
    "training runtime (minutes)": training_seconds / 60,
    "selected epoch": best_epoch,
    "selected epoch runtime (minutes)": float(
        training_history.loc[training_history["epoch"].eq(best_epoch), "epoch_seconds"].iloc[0] / 60
    ),
    "peak allocated GPU memory (GiB)": peak_gpu_memory_gib,
    "total parameters": total_parameters,
    "trainable parameters": trainable_parameters,
    "trainable percent": 100 * trainable_parameters / total_parameters,
    "GPU": environment_info["GPU"],
}
display(pd.Series(resource_summary, name="Value").to_frame())

## 27. Save Final Artefacts

The PEFT adapter already contains the persisted `lm_head` because it was registered in `modules_to_save`. The head is also written separately for auditability. The remaining files below make feature order, text conversion, split membership, threshold, metrics and runtime explicit for a later notebook.

In [ ]:
def save_experiment_config(path: Path) -> None:
    "Save the configuration needed to interpret and reproduce this run."
    config = {
        "run_id": RUN_ID,
        "model_id": MODEL_ID,
        "input_source": "GitHub clone",
        "repository_input_paths": {
            name: path.relative_to(REPO_ROOT).as_posix()
            for name, path in REQUIRED_INPUTS.items()
        },
        "qwen_output_root": str(QWEN_OUTPUT_ROOT),
        "target": TARGET,
        "classes": {"0": "no acute rejection within 30 days", "1": "acute rejection within 30 days"},
        "feature_count": len(FEATURES),
        "serialisation_version": SERIALISATION_VERSION,
        "max_seq_length": MAX_SEQ_LENGTH,
        "training": training_configuration,
        "best_epoch": best_epoch,
        "selected_threshold": SELECTED_THRESHOLD,
        "selection_data": "validation only",
        "test_evaluations": 1,
        "machine_unlearning_performed": False,
    }
    path.write_text(json.dumps(config, indent=2), encoding="utf-8")


(RUN_RESULT_DIR / "feature_order.json").write_text(json.dumps(FEATURES, indent=2), encoding="utf-8")
(RUN_RESULT_DIR / "serialisation_specification.json").write_text(
    json.dumps({
        "version": SERIALISATION_VERSION,
        "feature_order": FEATURES,
        "display_labels": FEATURE_LABELS,
        "template": "{display_label}: {deterministically_formatted_value}.",
        "separator": "newline",
        "target_included": False,
        "identifiers_included": False,
    }, indent=2),
    encoding="utf-8",
)
split_assignments.to_csv(RUN_RESULT_DIR / "frozen_split_memberships.csv", index=False)
pd.DataFrame([{
    "validation_loss": validation_result["loss"],
    "validation_pr_auc": validation_result["pr_auc"],
    "selected_threshold": SELECTED_THRESHOLD,
}]).to_csv(RUN_RESULT_DIR / "validation_metrics.csv", index=False)
pd.DataFrame([resource_summary]).to_csv(RUN_RESULT_DIR / "runtime_resources.csv", index=False)
comparison.to_csv(RUN_RESULT_DIR / "mlp_qwen_test_comparison.csv", index=False)
save_experiment_config(RUN_RESULT_DIR / "experiment_configuration.json")

artefact_manifest = {
    "best_adapter_and_head": str(BEST_ADAPTER_DIR),
    "separate_head_state": str(RUN_MODEL_DIR / "binary_classification_head.pt"),
    "tokenizer_processor": str(BEST_ADAPTER_DIR),
    "configuration": str(RUN_RESULT_DIR / "experiment_configuration.json"),
    "feature_order": str(RUN_RESULT_DIR / "feature_order.json"),
    "serialisation": str(RUN_RESULT_DIR / "serialisation_specification.json"),
    "threshold": str(RUN_RESULT_DIR / "selected_threshold.json"),
    "split_memberships": str(RUN_RESULT_DIR / "frozen_split_memberships.csv"),
    "training_history": str(RUN_RESULT_DIR / "training_history.csv"),
    "validation_metrics": str(RUN_RESULT_DIR / "validation_metrics.csv"),
    "test_metrics": str(RUN_RESULT_DIR / "test_metrics.csv"),
    "test_predictions": str(RUN_RESULT_DIR / "test_predictions.csv"),
    "runtime": str(RUN_RESULT_DIR / "runtime_resources.csv"),
}
(RUN_RESULT_DIR / "artefact_manifest.json").write_text(
    json.dumps(artefact_manifest, indent=2), encoding="utf-8"
)
display(pd.Series(artefact_manifest, name="Saved path").to_frame())

## 28. Findings

The statements below are generated from this run rather than written in advance. “Learned signal” means that test AUROC exceeds 0.5 and PR-AUC exceeds test prevalence; it is not a claim of clinical usefulness.

In [ ]:
finite_training = bool(np.isfinite(training_history[[
    "training_loss", "validation_loss", "validation_pr_auc"
]].to_numpy()).all())
learned_signal = bool(
    test_metrics["auroc"] > 0.5 and test_metrics["pr_auc"] > test_metrics["prevalence"]
)
competitive_with_mlp = qwen_wins >= 3
practical_on_colab = finite_training and peak_gpu_memory_gib < environment_info["GPU memory (GiB)"]

display(Markdown(
    f"- **Learned the task:** {'yes' if learned_signal else 'not established'} "
    f"(test AUROC {test_metrics['auroc']:.4f}; PR-AUC {test_metrics['pr_auc']:.4f}; "
    f"prevalence {test_metrics['prevalence']:.4f}).\n"
    f"- **Competitive with the MLP:** {'yes on the majority of listed metrics' if competitive_with_mlp else 'no on the majority of listed metrics'}.\n"
    f"- **Training stability:** {'all tracked values remained finite' if finite_training else 'a non-finite value was recorded'}.\n"
    f"- **Colab/Unsloth practicality:** {'the run fit the available GPU' if practical_on_colab else 'not established by this run'}; "
    f"training took {training_seconds / 60:.1f} minutes."
))

## 29. Limitations

- Qwen3.5-2B is much larger and more computationally expensive than the MLP.
- Tabular fields must be converted to text, which may be less natural than direct numeric/categorical processing.
- Only one model size and one conservative training configuration are evaluated.
- A single Colab run does not establish variance across seeds or GPU types.
- This baseline does not test deletion or forgetting.
- The dataset is synthetic, so results do not establish clinical generalisability.

## 30. Readiness for Machine-Unlearning Extension

A later notebook may study Hospital Removal, Full Retraining, Gradient Difference, Truth Ratio, KS comparison, retained utility and runtime **only if** this baseline produces finite training, non-degenerate predictions and useful ranking signal. Those methods are not executed here.

In [ ]:
READY_FOR_UNLEARNING_EXTENSION = bool(
    finite_training
    and learned_signal
    and len(np.unique(test_predictions["prediction"])) == 2
    and RELOAD_VERIFIED
)

display(Markdown(
    f"**Decision: {'ready for a separately designed machine-unlearning extension' if READY_FOR_UNLEARNING_EXTENSION else 'not yet ready for machine-unlearning'}**. "
    "A later notebook would reuse the Base model identifier, best LoRA adapter, saved two-class head, "
    "tokenizer, feature order, serialisation specification, frozen split memberships, validation threshold "
    "and baseline predictions saved above. No forgetting result is implied by this decision."
))

## Final Verification

These checks make the notebook's methodological contract explicit. A failed assertion should be investigated rather than bypassed.

In [ ]:
MACHINE_UNLEARNING_PERFORMED = False
completion_checks = pd.DataFrame([
    {"check": "CUDA was used", "pass": DEVICE.type == "cuda" and torch.cuda.is_available()},
    {"check": "Exact Qwen3.5-2B model used", "pass": MODEL_ID == "unsloth/Qwen3.5-2B-Base"},
    {"check": "Required inputs came from GitHub clone", "pass": all(path.is_relative_to(REPO_ROOT) for path in REQUIRED_INPUTS.values())},
    {"check": "Qwen outputs use Google Drive", "pass": QWEN_OUTPUT_ROOT.is_relative_to(Path("/content/drive/MyDrive")) and not QWEN_OUTPUT_ROOT.is_relative_to(REPO_ROOT)},
    {"check": "Original frozen splits reused", "pass": ORIGINAL_SPLITS_REUSED},
    {"check": "No target or identifier leakage", "pass": NO_TARGET_LEAKAGE},
    {"check": "Exactly two logits produced", "pass": LOGITS_SHAPE_VERIFIED},
    {"check": "Classification head was trainable", "pass": head_was_trainable},
    {"check": "Test rows used for training = 0", "pass": TEST_ROWS_USED_FOR_TRAINING == 0},
    {"check": "Best checkpoint exists", "pass": BEST_ADAPTER_DIR.exists()},
    {"check": "Separate binary head state exists", "pass": (RUN_MODEL_DIR / "binary_classification_head.pt").exists()},
    {"check": "Frozen threshold exists", "pass": (RUN_RESULT_DIR / "selected_threshold.json").exists()},
    {"check": "Final metrics saved", "pass": (RUN_RESULT_DIR / "test_metrics.csv").exists()},
    {"check": "Saved classifier reload verified", "pass": RELOAD_VERIFIED},
    {"check": "Machine unlearning not performed", "pass": not MACHINE_UNLEARNING_PERFORMED},
])
assert completion_checks["pass"].all(), completion_checks.loc[~completion_checks["pass"]]
display(completion_checks)
print(f"Notebook 09 baseline complete. Persistent results: {RUN_RESULT_DIR}")